In [2]:
import time
import random
import heapq
import math
from collections import deque

class MazeSolver:
    def __init__(self, file_path):
        self.matrix = self._load_maze(file_path)
        self.rows = len(self.matrix)
        self.cols = len(self.matrix[0])
        self.starts = self._find_positions('2')
        self.goals = self._find_positions('3')

    def _load_maze(self, file_path):
        with open(file_path, 'r') as f:
            return [list(line.strip()) for line in f.readlines()]

    def _find_positions(self, char):
        return [(r, c) for r in range(self.rows) for c in range(self.cols) if self.matrix[r][c] == char]

    def get_neighbors(self, pos):
        r, c = pos
        neighbors = []
        # Movimientos: Arriba, Abajo, Izquierda, Derecha
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < self.rows and 0 <= nc < self.cols and self.matrix[nr][nc] != '1':
                neighbors.append((nr, nc))
        return neighbors

    # --- Heurísticas ---
    @staticmethod
    def manhattan(a, b):
        return abs(a[0] - b[0]) + abs(a[1] - b[1])

    @staticmethod
    def euclidean(a, b):
        return math.sqrt((a[0] - b[0])**2 + (a[1] - b[1])**2)

    # --- Cálculo de Branching Factor Efectivo (b*) ---
    def calculate_ebf(self, nodes_explored, depth):
        if depth == 0: return 0
        # Una aproximación simple: N = b*^d => b* = N^(1/d)
        return round(nodes_explored**(1/depth), 4)

    # --- Algoritmos de Búsqueda ---
    def search(self, start_pos, algorithm="BFS", heuristic_type=None):
        goal = self.goals[0]
        start_time = time.time()
        
        # Estructura: (posicion, path) o (prioridad, posicion, path)
        explored = set()
        nodes_count = 0
        
        if algorithm == "BFS":
            frontier = deque([(start_pos, [])])
        elif algorithm == "DFS":
            frontier = [(start_pos, [])]
        else: # Informed (A* or Greedy)
            frontier = [(0, start_pos, [])]

        while frontier:
            if algorithm == "BFS":
                current, path = frontier.popleft()
            elif algorithm == "DFS":
                current, path = frontier.pop()
            else:
                _, current, path = heapq.heappop(frontier)

            if current in explored: continue
            
            nodes_count += 1
            explored.add(current)

            if current == goal:
                end_time = time.time()
                full_path = path + [current]
                return {
                    "algorithm": f"{algorithm} ({heuristic_type if heuristic_type else 'N/A'})",
                    "path_len": len(full_path),
                    "nodes_explored": nodes_count,
                    "time": end_time - start_time,
                    "ebf": self.calculate_ebf(nodes_count, len(full_path))
                }

            for neighbor in self.get_neighbors(current):
                if neighbor not in explored:
                    new_path = path + [current]
                    if algorithm == "BFS" or algorithm == "DFS":
                        frontier.append((neighbor, new_path))
                    else:
                        h = self.manhattan(neighbor, goal) if heuristic_type == "manhattan" else self.euclidean(neighbor, goal)
                        g = len(new_path)
                        priority = g + h if algorithm == "A*" else h
                        heapq.heappush(frontier, (priority, neighbor, new_path))
        
        return None

# --- Función Principal para ejecutar los casos ---
def run_experiment(file_path):
    solver = MazeSolver(file_path)
    
    # Definición de casos
    cases = {
        "Caso 1 (Único Inicio)": solver.starts[0],
        "Caso 2 (Inicio Aleatorio de '2')": random.choice(solver.starts),
        "Caso 3 (Inicio Aleatorio en camino '0')": random.choice(solver._find_positions('0'))
    }

    algos = [
        ("BFS", None), 
        ("DFS", None), 
        ("A*", "manhattan"), 
        ("A*", "euclidean"), 
        ("Greedy", "manhattan"), 
        ("Greedy", "euclidean")
    ]

    for case_name, start_node in cases.items():
        print(f"\n--- {case_name} en {start_node} ---")
        print(f"{'Algoritmo':<30} | {'Camino':<7} | {'Nodos':<7} | {'Tiempo (s)':<10} | {'EBF':<7}")
        print("-" * 80)
        
        for name, heuristic in algos:
            res = solver.search(start_node, name, heuristic)
            if res:
                print(f"{res['algorithm']:<30} | {res['path_len']:<7} | {res['nodes_explored']:<7} | {res['time']:<10.5f} | {res['ebf']:<7}")
            else:
                print(f"{name:<30} | No se encontró solución")

In [6]:
run_experiment('test_maze.txt')


--- Caso 1 (Único Inicio) en (1, 1) ---
Algoritmo                      | Camino  | Nodos   | Tiempo (s) | EBF    
--------------------------------------------------------------------------------
BFS (N/A)                      | 129     | 665     | 0.00208    | 1.0517 
DFS (N/A)                      | 185     | 516     | 0.00201    | 1.0343 
A* (manhattan)                 | 129     | 534     | 0.00200    | 1.0499 
A* (euclidean)                 | 129     | 598     | 0.00251    | 1.0508 
Greedy (manhattan)             | 153     | 306     | 0.00251    | 1.0381 
Greedy (euclidean)             | 133     | 403     | 0.00201    | 1.0461 

--- Caso 2 (Inicio Aleatorio de '2') en (1, 1) ---
Algoritmo                      | Camino  | Nodos   | Tiempo (s) | EBF    
--------------------------------------------------------------------------------
BFS (N/A)                      | 129     | 665     | 0.00251    | 1.0517 
DFS (N/A)                      | 185     | 516     | 0.00100    | 1.0343 
A* (m

In [29]:
run_experiment('Prueba_1.txt')


--- Caso 1 (Único Inicio) en (1, 1) ---
Algoritmo                      | Camino  | Nodos   | Tiempo (s) | EBF    
--------------------------------------------------------------------------------
BFS (N/A)                      | 120     | 1846    | 0.01054    | 1.0647 
DFS (N/A)                      | 456     | 589     | 0.01105    | 1.0141 
A* (manhattan)                 | 120     | 986     | 0.00531    | 1.0591 
A* (euclidean)                 | 120     | 989     | 0.01153    | 1.0592 
Greedy (manhattan)             | 120     | 120     | 0.00000    | 1.0407 
Greedy (euclidean)             | 142     | 177     | 0.00200    | 1.0371 

--- Caso 2 (Inicio Aleatorio de '2') en (1, 1) ---
Algoritmo                      | Camino  | Nodos   | Tiempo (s) | EBF    
--------------------------------------------------------------------------------
BFS (N/A)                      | 120     | 1846    | 0.01073    | 1.0647 
DFS (N/A)                      | 456     | 589     | 0.00391    | 1.0141 
A* (m

In [12]:
run_experiment('Prueba_2.txt')


--- Caso 1 (Único Inicio) en (1, 1) ---
Algoritmo                      | Camino  | Nodos   | Tiempo (s) | EBF    
--------------------------------------------------------------------------------
BFS                            | No se encontró solución
DFS                            | No se encontró solución
A*                             | No se encontró solución
A*                             | No se encontró solución
Greedy                         | No se encontró solución
Greedy                         | No se encontró solución

--- Caso 2 (Inicio Aleatorio de '2') en (1, 1) ---
Algoritmo                      | Camino  | Nodos   | Tiempo (s) | EBF    
--------------------------------------------------------------------------------
BFS                            | No se encontró solución
DFS                            | No se encontró solución
A*                             | No se encontró solución
A*                             | No se encontró solución
Greedy                     

In [23]:
run_experiment('Prueba_3.txt')


--- Caso 1 (Único Inicio) en (61, 60) ---
Algoritmo                      | Camino  | Nodos   | Tiempo (s) | EBF    
--------------------------------------------------------------------------------
BFS                            | No se encontró solución
DFS                            | No se encontró solución
A*                             | No se encontró solución
A*                             | No se encontró solución
Greedy                         | No se encontró solución
Greedy                         | No se encontró solución

--- Caso 2 (Inicio Aleatorio de '2') en (61, 60) ---
Algoritmo                      | Camino  | Nodos   | Tiempo (s) | EBF    
--------------------------------------------------------------------------------
BFS                            | No se encontró solución
DFS                            | No se encontró solución
A*                             | No se encontró solución
A*                             | No se encontró solución
Greedy                 

In [31]:
run_experiment('Laberinto1-1.txt')


--- Caso 1 (Único Inicio) en (28, 114) ---
Algoritmo                      | Camino  | Nodos   | Tiempo (s) | EBF    
--------------------------------------------------------------------------------
BFS (N/A)                      | 109     | 10195   | 0.04873    | 1.0884 
DFS (N/A)                      | 2903    | 16115   | 0.58970    | 1.0033 
A* (manhattan)                 | 109     | 742     | 0.00000    | 1.0625 
A* (euclidean)                 | 109     | 1251    | 0.02210    | 1.0676 
Greedy (manhattan)             | 117     | 118     | 0.00118    | 1.0416 
Greedy (euclidean)             | 125     | 239     | 0.00000    | 1.0448 

--- Caso 2 (Inicio Aleatorio de '2') en (28, 114) ---
Algoritmo                      | Camino  | Nodos   | Tiempo (s) | EBF    
--------------------------------------------------------------------------------
BFS (N/A)                      | 109     | 10195   | 0.02947    | 1.0884 
DFS (N/A)                      | 2903    | 16115   | 0.59305    | 1.0033 

In [33]:
run_experiment('Laberinto2-1.txt')


--- Caso 1 (Único Inicio) en (28, 114) ---
Algoritmo                      | Camino  | Nodos   | Tiempo (s) | EBF    
--------------------------------------------------------------------------------
BFS (N/A)                      | 109     | 10217   | 0.06046    | 1.0884 
DFS (N/A)                      | 1519    | 1519    | 0.03318    | 1.0048 
A* (manhattan)                 | 109     | 696     | 0.00254    | 1.0619 
A* (euclidean)                 | 109     | 1206    | 0.00590    | 1.0673 
Greedy (manhattan)             | 119     | 121     | 0.00000    | 1.0411 
Greedy (euclidean)             | 197     | 1187    | 0.00462    | 1.0366 

--- Caso 2 (Inicio Aleatorio de '2') en (28, 114) ---
Algoritmo                      | Camino  | Nodos   | Tiempo (s) | EBF    
--------------------------------------------------------------------------------
BFS (N/A)                      | 109     | 10217   | 0.03707    | 1.0884 
DFS (N/A)                      | 1519    | 1519    | 0.03190    | 1.0048 

In [36]:
run_experiment('Laberinto3-1.txt')


--- Caso 1 (Único Inicio) en (28, 114) ---
Algoritmo                      | Camino  | Nodos   | Tiempo (s) | EBF    
--------------------------------------------------------------------------------
BFS (N/A)                      | 109     | 9885    | 0.05352    | 1.0881 
DFS (N/A)                      | 1279    | 20169   | 0.60254    | 1.0078 
A* (manhattan)                 | 109     | 645     | 0.00316    | 1.0611 
A* (euclidean)                 | 109     | 1079    | 0.00129    | 1.0662 
Greedy (manhattan)             | 117     | 118     | 0.00000    | 1.0416 
Greedy (euclidean)             | 109     | 138     | 0.00000    | 1.0462 

--- Caso 2 (Inicio Aleatorio de '2') en (28, 114) ---
Algoritmo                      | Camino  | Nodos   | Tiempo (s) | EBF    
--------------------------------------------------------------------------------
BFS (N/A)                      | 109     | 9885    | 0.04008    | 1.0881 
DFS (N/A)                      | 1279    | 20169   | 0.89953    | 1.0078 

In [39]:
run_experiment('Laberinto1-2.txt')


--- Caso 1 (Único Inicio) en (28, 57) ---
Algoritmo                      | Camino  | Nodos   | Tiempo (s) | EBF    
--------------------------------------------------------------------------------
BFS (N/A)                      | 652     | 4755    | 0.04050    | 1.0131 
DFS (N/A)                      | 2584    | 3350    | 0.09734    | 1.0031 
A* (manhattan)                 | 652     | 3736    | 0.05006    | 1.0127 
A* (euclidean)                 | 652     | 3849    | 0.03967    | 1.0127 
Greedy (manhattan)             | 656     | 2212    | 0.01360    | 1.0118 
Greedy (euclidean)             | 672     | 2216    | 0.02266    | 1.0115 

--- Caso 2 (Inicio Aleatorio de '2') en (28, 57) ---
Algoritmo                      | Camino  | Nodos   | Tiempo (s) | EBF    
--------------------------------------------------------------------------------
BFS (N/A)                      | 652     | 4755    | 0.02936    | 1.0131 
DFS (N/A)                      | 2584    | 3350    | 0.07152    | 1.0031 
A

In [38]:
run_experiment('Laberinto2-2.txt')


--- Caso 1 (Único Inicio) en (28, 57) ---
Algoritmo                      | Camino  | Nodos   | Tiempo (s) | EBF    
--------------------------------------------------------------------------------
BFS (N/A)                      | 316     | 4103    | 0.01728    | 1.0267 
DFS (N/A)                      | 1288    | 1551    | 0.01858    | 1.0057 
A* (manhattan)                 | 316     | 3299    | 0.02923    | 1.026  
A* (euclidean)                 | 316     | 3768    | 0.01932    | 1.0264 
Greedy (manhattan)             | 378     | 2495    | 0.00477    | 1.0209 
Greedy (euclidean)             | 354     | 2427    | 0.02364    | 1.0223 

--- Caso 2 (Inicio Aleatorio de '2') en (28, 57) ---
Algoritmo                      | Camino  | Nodos   | Tiempo (s) | EBF    
--------------------------------------------------------------------------------
BFS (N/A)                      | 316     | 4103    | 0.02751    | 1.0267 
DFS (N/A)                      | 1288    | 1551    | 0.01871    | 1.0057 
A

In [37]:
run_experiment('Laberinto3-2.txt')


--- Caso 1 (Único Inicio) en (28, 57) ---
Algoritmo                      | Camino  | Nodos   | Tiempo (s) | EBF    
--------------------------------------------------------------------------------
BFS (N/A)                      | 184     | 3789    | 0.02393    | 1.0458 
DFS (N/A)                      | 1184    | 3166    | 0.04074    | 1.0068 
A* (manhattan)                 | 184     | 2040    | 0.00825    | 1.0423 
A* (euclidean)                 | 184     | 2469    | 0.01010    | 1.0434 
Greedy (manhattan)             | 188     | 1358    | 0.00000    | 1.0391 
Greedy (euclidean)             | 188     | 610     | 0.01406    | 1.0347 

--- Caso 2 (Inicio Aleatorio de '2') en (28, 57) ---
Algoritmo                      | Camino  | Nodos   | Tiempo (s) | EBF    
--------------------------------------------------------------------------------
BFS (N/A)                      | 184     | 3789    | 0.01407    | 1.0458 
DFS (N/A)                      | 1184    | 3166    | 0.04183    | 1.0068 
A